In [1]:
# ProjectTwoDashboard.ipynb
from jupyter_dash import JupyterDash
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
import pandas as pd
import base64
import os
from animal_shelter import AnimalShelter # Ensure this file is in the same directory

# Database connection parameters
username = "aacuser"
password = "SNHU1234"
host = "nv-desktop-services.apporto.com"
port = 30261

# Create an instance of the AnimalShelter class
try:
    db = AnimalShelter(username, password, host, port)
    print("AnimalShelter instance created successfully.")
except Exception as e:
    print(f"Failed to create AnimalShelter instance: {e}")
    # Exit or handle the error appropriately if the database connection is critical
    exit()

# Read all data from MongoDB initially
# This DataFrame will serve as the base for filtering
try:
    df = pd.DataFrame.from_records(db.read({}))
    if '_id' in df.columns:
        df.drop(columns=['_id'], inplace=True)
    print("Initial DataFrame loaded successfully.")
except Exception as e:
    print(f"Failed to load initial data from MongoDB: {e}")
    df = pd.DataFrame() # Initialize an empty DataFrame to prevent further errors

# Check the DataFrame columns and content for debugging
print("DataFrame Columns (initial load):", df.columns.tolist())
print("DataFrame Head (initial load):\n", df.head())

# Initialize the Dash app
app = JupyterDash(__name__)

# Load Grazioso Salvare logo
image_filename = 'Grazioso_Salvare_Logo.png'
encoded_image = None
if os.path.isfile(image_filename):
    with open(image_filename, 'rb') as f:
        encoded_image = base64.b64encode(f.read()).decode('ascii')
    print(f"Logo '{image_filename}' loaded successfully.")
else:
    print(f"Warning: Logo file not found at '{image_filename}'. Please ensure it's in the correct directory.")

# Dashboard layout
app.layout = html.Div([
    html.Center(html.B(html.H1('CS-340 Dashboard - Grazioso Salvare'))),
    html.Div([
        html.A(
            html.Img(
                src=f'data:image/png;base64,{encoded_image}' if encoded_image else '',
                style={'width': '200px', 'height': 'auto', 'marginRight': '20px'}
            ),
            href="http://www.snhu.edu", # Grazioso Salvare's requested URL
            target="_blank" # Open in new tab
        ),
        html.Div(
            html.P("Dashboard Creator: Saad Ohiduzzaman"), # Unique identifier
            style={'display': 'inline-block', 'verticalAlign': 'top', 'marginTop': '20px'}
        )
    ], style={'display': 'flex', 'alignItems': 'center', 'justifyContent': 'center'}),
    html.Hr(),
    html.H2("Filter Dogs by Rescue Type:"),
    dcc.RadioItems(
        id='filter-type',
        options=[
            {'label': 'Water Rescue', 'value': 'Water Rescue'},
            {'label': 'Mountain or Wilderness Rescue', 'value': 'Mountain or Wilderness Rescue'},
            {'label': 'Disaster or Individual Tracking', 'value': 'Disaster or Individual Tracking'},
            {'label': 'Reset', 'value': 'Reset'}
        ],
        value='Reset', # Default to Reset to show all data initially
        inline=True,
        style={'margin': '10px'}
    ),
    html.Hr(),
    html.H2("Animal Data Table:"),
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i} for i in df.columns if i not in ['location_lat', 'location_long']], # Exclude lat/long from table
        data=df.to_dict('records'),
        page_size=10,
        style_table={'overflowX': 'auto', 'margin': '10px'},
        style_cell={'textAlign': 'left', 'padding': '5px'},
        filter_action='native',
        sort_action='native',
        row_selectable='single',
        selected_rows=[],
    ),
    html.Hr(),
    html.H2("Charts and Map:"),
    html.Div([
        dcc.Graph(id='pie-chart-id', style={'display': 'inline-block', 'width': '49%'}),
        html.Div(id='map-id', style={'display': 'inline-block', 'width': '49%', 'verticalAlign': 'top'})
    ], style={'display': 'flex', 'justifyContent': 'space-around'})
])

# Callback for updating the data table based on filter selection
@app.callback(Output('datatable-id', 'data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    query = {} # Base query for MongoDB

    if filter_type == 'Water Rescue':
        query = {
            "breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }
    elif filter_type == 'Mountain or Wilderness Rescue':
        query = {
            "breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }
    elif filter_type == 'Disaster or Individual Tracking':
        query = {
            "breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }
    elif filter_type == 'Reset':
        query = {} # Empty query to retrieve all documents

    # Fetch data from MongoDB using the AnimalShelter's read method
    try:
        filtered_data = db.read(query)
        filtered_df = pd.DataFrame.from_records(filtered_data)
        if '_id' in filtered_df.columns:
            filtered_df.drop(columns=['_id'], inplace=True)
        print(f"Filtered DataFrame for '{filter_type}' has {len(filtered_df)} rows.")
    except Exception as e:
        print(f"Error fetching data for filter '{filter_type}': {e}")
        filtered_df = pd.DataFrame() # Return empty DataFrame on error

    if filtered_df.empty:
        print(f"Filtered DataFrame is empty for filter type: {filter_type}")
        return []
    
    return filtered_df.to_dict('records')

# Callback for updating the pie chart
@app.callback(Output('pie-chart-id', "figure"), # Changed output to 'figure' for dcc.Graph
              [Input('datatable-id', "derived_virtual_data")])
def update_pie_chart(viewData):
    if viewData is None or len(viewData) == 0:
        # Return an empty figure or a placeholder if no data
        return px.pie(title='No data available for the selected filter.')
    
    dff = pd.DataFrame.from_dict(viewData)
    
    # Check if 'breed' column exists and is not empty
    if 'breed' not in dff.columns or dff['breed'].empty:
        return px.pie(title='No breed data available for charting.')
    
    # Create a pie chart with customized size and layout
    pie_chart = px.pie(dff, names='breed', title='Distribution of Breeds', hole=0.3)
    pie_chart.update_traces(textinfo='percent+label')
    pie_chart.update_layout(width=500, height=400, margin=dict(l=20, r=20, t=40, b=20)) # Adjust margins
    
    return pie_chart

# Callback for updating the map
@app.callback(Output('map-id', "children"),
              [Input('datatable-id', "derived_virtual_data"),
               Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):
    if viewData is None or len(viewData) == 0 or index is None or len(index) == 0:
        return html.Div("Select a row in the table to view its location on the map.",
                        style={'textAlign': 'center', 'marginTop': '50px'})
    
    dff = pd.DataFrame.from_dict(viewData)
    
    # Ensure latitude and longitude columns exist and are numeric
    if 'location_lat' not in dff.columns or 'location_long' not in dff.columns:
        return html.Div("Geolocation data (latitude/longitude) not available in the dataset.",
                        style={'textAlign': 'center', 'marginTop': '50px'})
    
    # Convert to numeric, coercing errors to NaN
    dff['location_lat'] = pd.to_numeric(dff['location_lat'], errors='coerce')
    dff['location_long'] = pd.to_numeric(dff['location_long'], errors='coerce')

    # Drop rows where lat/long are NaN after conversion
    dff.dropna(subset=['location_lat', 'location_long'], inplace=True)

    if dff.empty:
        return html.Div("No valid geolocation data after filtering.",
                        style={'textAlign': 'center', 'marginTop': '50px'})

    row_data = dff.iloc[index[0]] # Get the selected row's data

    # Check if the specific row has valid lat/long
    if pd.isna(row_data['location_lat']) or pd.isna(row_data['location_long']):
        return html.Div("Selected row has invalid geolocation data.",
                        style={'textAlign': 'center', 'marginTop': '50px'})

    # Default center for Austin, TX
    map_center = [30.2672, -97.7431] # Austin coordinates
    
    # Use the selected animal's location if available, otherwise use default center
    marker_position = [row_data['location_lat'], row_data['location_long']] if not pd.isna(row_data['location_lat']) and not pd.isna(row_data['location_long']) else map_center

    # Get animal name for popup/tooltip
    animal_name = row_data.get('name', 'N/A') # Use .get() for safer access
    animal_type = row_data.get('animal_type', 'N/A')

    return [
        dl.Map(style={'width': '100%', 'height': '400px', 'margin': '10px'}, center=map_center, zoom=10, children=[
            dl.TileLayer(),
            dl.Marker(position=marker_position, children=[
                dl.Tooltip(f"Type: {animal_type}, Name: {animal_name}"),
                dl.Popup([
                    html.H3(f"Animal: {animal_name}"),
                    html.P(f"Type: {animal_type}"),
                    html.P(f"Breed: {row_data.get('breed', 'N/A')}")
                ])
            ])
        ])
    ]

# Run the server
if __name__ == '__main__':
    app.run_server(port=8051, debug=True)



Connected successfully to MongoDB
AnimalShelter instance created successfully.
Found 9975 document(s) matching the query.
Initial DataFrame loaded successfully.
DataFrame Columns (initial load): ['rec_num', 'age_upon_outcome', 'animal_id', 'animal_type', 'breed', 'color', 'date_of_birth', 'datetime', 'monthyear', 'name', 'outcome_subtype', 'outcome_type', 'sex_upon_outcome', 'location_lat', 'location_long', 'age_upon_outcome_in_weeks']
DataFrame Head (initial load):
    rec_num age_upon_outcome animal_id animal_type                    breed  \
0        1          3 years   A746874         Cat   Domestic Shorthair Mix   
1        9          3 years   A720214         Dog   Labrador Retriever Mix   
2        3          2 years   A716330         Dog  Chihuahua Shorthair Mix   
3       11           1 year   A721199         Dog   Dachshund Wirehair Mix   
4       10         3 months   A664290         Cat   Domestic Shorthair Mix   

         color date_of_birth             datetime          